In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torchvision.datasets import Flowers102
from torch.utils.data import DataLoader, Subset, ConcatDataset
import numpy as np
import matplotlib.pyplot as plt
import random

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = Flowers102(root='data', split='train', transform=transform, download=True)
val_dataset = Flowers102(root='data', split='val', transform=transform, download=True)
test_dataset = Flowers102(root='data', split='test', transform=transform, download=True)

# Combine train and val for active learning pool
full_dataset = ConcatDataset([train_dataset, val_dataset])

def get_model():
    model = models.resnet50(pretrained=True)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 102)
    return model


def random_sampling(unlabeled_indices, num_samples):
    return random.sample(unlabeled_indices, num_samples)

def uncertainty_sampling(model, unlabeled_loader, unlabeled_indices, num_samples, device):
    model.eval()
    uncertainties = []

    with torch.no_grad():
        for inputs, _ in unlabeled_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=1)
            uncertainties.extend(entropy.cpu().numpy())

    # Select indices with highest entropy
    uncertainties = np.array(uncertainties)
    selected_indices = np.argsort(uncertainties)[-num_samples:]
    return [unlabeled_indices[i] for i in selected_indices]

def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()

def bayesian_active_learning(model, unlabeled_loader, unlabeled_indices, num_samples, device, mc_iterations=10):
    model.eval()
    enable_dropout(model)
    all_probs = []

    with torch.no_grad():
        for _ in range(mc_iterations):
            probs = []
            for inputs, _ in unlabeled_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                prob = F.softmax(outputs, dim=1)
                probs.append(prob.cpu().numpy())
            all_probs.append(np.concatenate(probs, axis=0))

    mean_probs = np.mean(all_probs, axis=0)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
    selected_indices = np.argsort(entropy)[-num_samples:]
    return [unlabeled_indices[i] for i in selected_indices]


def hybrid_sampling(model, unlabeled_loader, unlabeled_indices, labeled_dataset, num_samples, device):
    model.eval()
    uncertainties = []

    with torch.no_grad():
        for inputs, _ in unlabeled_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=1)
            uncertainties.extend(entropy.cpu().numpy())

    # Compute diversity (distance from labeled data mean)
    unlabeled_features = []
    for inputs, _ in unlabeled_loader:
        inputs = inputs.to(device)
        features = model.avgpool(model.layer4(model.layer3(model.layer2(model.layer1(model.relu(model.bn1(model.conv1(inputs))))))))
        features = features.view(features.size(0), -1)
        unlabeled_features.extend(features.cpu().numpy())

    labeled_loader = DataLoader(labeled_dataset, batch_size=64, shuffle=False)
    labeled_features = []
    with torch.no_grad():
        for inputs, _ in labeled_loader:
            inputs = inputs.to(device)
            features = model.avgpool(model.layer4(model.layer3(model.layer2(model.layer1(model.relu(model.bn1(model.conv1(inputs))))))))
            features = features.view(features.size(0), -1)
            labeled_features.extend(features.cpu().numpy())

    labeled_mean = np.mean(labeled_features, axis=0)
    diversity = np.linalg.norm(np.array(unlabeled_features) - labeled_mean, axis=1)

    # Combine uncertainty and diversity
    hybrid_score = 0.5 * np.array(uncertainties) + 0.5 * diversity
    selected_indices = np.argsort(hybrid_score)[-num_samples:]
    return [unlabeled_indices[i] for i in selected_indices]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
initial_labeled_samples = 100
query_batch_size = 50
num_iterations = 10
epochs_per_round = 5

# Initialize indices
all_indices = list(range(len(full_dataset)))
labeled_indices = random.sample(all_indices, initial_labeled_samples)
unlabeled_indices = list(set(all_indices) - set(labeled_indices))

# Initialize datasets
labeled_dataset = Subset(full_dataset, labeled_indices)
unlabeled_dataset = Subset(full_dataset, unlabeled_indices)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Active learning iterations
for iteration in range(num_iterations):
    print(f"--- Iteration {iteration + 1} ---")
    model = get_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Train the model
    train_loader = DataLoader(labeled_dataset, batch_size=32, shuffle=True)
    model.train()
    for epoch in range(epochs_per_round):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")

    # Select new samples (e.g., using uncertainty sampling)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=False)
    selected_indices = uncertainty_sampling(model, unlabeled_loader, unlabeled_indices, query_batch_size, device)

    # Update indices and datasets
    labeled_indices.extend(selected_indices)
    unlabeled_indices = list(set(unlabeled_indices) - set(selected_indices))
    labeled_dataset = Subset(full_dataset, labeled_indices)
    unlabeled_dataset = Subset(full_dataset, unlabeled_indices)


# Assuming you store accuracies in a list
plt.plot(range(1, num_iterations + 1), accuracies, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Test Accuracy')
plt.title('Active Learning Performance')
plt.grid()
plt.show()
